In [1]:
from pathlib import Path

project_root = Path(".").resolve().parent.parent.parent
assert project_root.name == "pep-compass"

In [7]:
import pandas as pd

parents = pd.read_csv(
    project_root
    / "results/apex_predictions/data/hydramp/veltri/all/veltri_negative.csv"
)
mutants = pd.read_csv(
    project_root
    / "results/apex_predictions/results/mutants/veltri_negative/all/veltri_negative_direction_threshold=0.001_token_threshold=0.1_jacobian_mode=approx.csv"
)

In [9]:
display(parents.head())
display(mutants.head())

,Name,Sequence,A. baumannii ATCC 19606,E. coli ATCC 11775,E. coli AIG221,E. coli AIG222,K. pneumoniae ATCC 13883,P. aeruginosa PA01,P. aeruginosa PA14,S. aureus ATCC 12600,...,B. ovatus ATCC8483,E. rectale ATCC33656,C. symbiosum,R. obeum,R. torques,E. coli Nissle,Salmonella enterica ATCC 9150 (BEIRES NR-515),Salmonella enterica (BEIRES NR-170),Salmonella enterica ATCC 9150 (BEIRES NR-174),L. monocytogenes ATCC 19111 (BEIRES NR-106)
0,UniRef50_A0QYG2,EGAVSGVEGLPSGSAL,138.511180,139.801180,137.902370,140.05830,138.364410,137.98553,136.774020,141.596900,...,408.06840,299.70337,142.26360,141.87341,134.63571,2463.8271,106.403625,1105.36780,646.91650,131.340040
1,UniRef50_Q9FKB0,ELIDRCIGSKIWVIMKGDKELVGILKG,89.528824,118.419390,119.541725,128.80423,100.891540,90.99354,102.825260,70.190704,...,252.05864,210.20215,129.42552,130.82336,113.55690,1169.5818,61.302850,516.47375,488.59735,83.926080
2,UniRef50_P33768,QNIQDRIKMYIKKEEEEPTNFKNPF,130.894900,115.837326,114.040790,131.67166,119.030320,124.43645,127.947815,118.429825,...,396.36917,304.71548,133.56018,129.68240,100.20509,1961.2029,79.325660,943.49840,455.85287,114.144104
3,UniRef50_O42888,GEGITPANIAISWAVKRGTSVLPKSVNESRIVS,134.440840,133.998600,130.942200,133.20078,139.556900,138.98576,135.156510,126.896560,...,425.77026,311.65692,138.77722,139.58517,133.83589,2635.2510,103.507120,1179.88680,699.80206,126.254190
4,UniRef50_P51657,GCSSGIGLHLAVRLASDRSQSFKVYATLRDLKSQGPLLEAARA,115.702190,121.632590,119.013330,127.29396,122.310265,120.87968,124.867210,95.488450,...,372.47662,279.35890,136.31871,129.34870,119.17818,2587.9010,87.836210,1226.76670,667.98175,109.741745


,mutant,position,parent,direction_significance_threshold,token_threshold,parent_name,parent_sequence,A. baumannii ATCC 19606,E. coli ATCC 11775,E. coli AIG221,...,B. ovatus ATCC8483,E. rectale ATCC33656,C. symbiosum,R. obeum,R. torques,E. coli Nissle,Salmonella enterica ATCC 9150 (BEIRES NR-515),Salmonella enterica (BEIRES NR-170),Salmonella enterica ATCC 9150 (BEIRES NR-174),L. monocytogenes ATCC 19111 (BEIRES NR-106)
0,EGAVSGVEGLPSGSAL,15,EGAVSGVEGLPSGSAL,0.001,0.1,UniRef50_A0QYG2,EGAVSGVEGLPSGSAL,138.51120,139.80118,137.90237,...,408.06845,299.70350,142.26358,141.87337,134.63573,2463.8271,106.403660,1105.3683,646.91644,131.34004
1,EGAVSGVEGLPSGSAN,15,EGAVSGVEGLPSGSAL,0.001,0.1,UniRef50_A0QYG2,EGAVSGVEGLPSGSAL,139.28995,140.26088,138.06827,...,425.31943,311.48676,143.79056,144.51553,138.30191,2539.0486,108.815346,1091.2412,676.61480,133.30217
2,EGAVSGVEGLPSGSAL,15,EGAVSGVEGLPSGSAL,0.001,0.1,UniRef50_A0QYG2,EGAVSGVEGLPSGSAL,138.51120,139.80118,137.90237,...,408.06845,299.70350,142.26358,141.87337,134.63573,2463.8271,106.403660,1105.3683,646.91644,131.34004
3,EGAVSGVEGLPSGSAN,15,EGAVSGVEGLPSGSAL,0.001,0.1,UniRef50_A0QYG2,EGAVSGVEGLPSGSAL,139.28995,140.26088,138.06827,...,425.31943,311.48676,143.79056,144.51553,138.30191,2539.0486,108.815346,1091.2412,676.61480,133.30217
4,EGAVSGVEGLPSGSAL,15,EGAVSGVEGLPSGSAL,0.001,0.1,UniRef50_A0QYG2,EGAVSGVEGLPSGSAL,138.51120,139.80118,137.90237,...,408.06845,299.70350,142.26358,141.87337,134.63573,2463.8271,106.403660,1105.3683,646.91644,131.34004


In [54]:
import numpy as np
from typing import Literal, Optional


def compute_diff(
    parents_df: pd.DataFrame,
    mutants_df: pd.DataFrame,
    match_col_parents: str,
    match_col_mutants: str,
    value_cols: list[str],
    value_preprocessing: Optional[Literal["log"]],
    add_relative: bool,
) -> pd.DataFrame:
    """
    Compute the difference between the mutants and parents for each value column.
    """
    merged = mutants_df.merge(
        parents_df,
        left_on=match_col_mutants,
        right_on=match_col_parents,
        suffixes=("_mutants", "_parents"),
    )
    if value_preprocessing is None:
        vprep_fn = lambda x: x
    elif value_preprocessing == "log":
        vprep_fn = np.log
    else:
        raise ValueError(f"Invalid value_preprocessing: {value_preprocessing}")
    if value_preprocessing is None:
        value_preprocessing = ""
    else:
        value_preprocessing = "_" + value_preprocessing

    for value_col in value_cols:
        merged[f"{value_col}{value_preprocessing}_diff"] = vprep_fn(
            merged[f"{value_col}_mutants"]
        ) - vprep_fn(merged[f"{value_col}_parents"])
        if add_relative:
            merged[f"{value_col}{value_preprocessing}_diff_relative"] = merged[
                f"{value_col}{value_preprocessing}_diff"
            ] / vprep_fn(merged[f"{value_col}_parents"])
    return merged

In [48]:
mic_bac_columns = parents.head().columns.tolist()[2:]

In [49]:
len(mic_bac_columns)

34

In [52]:
diff = compute_diff(
    parents_df=parents,
    mutants_df=mutants,
    match_col_parents="Name",
    match_col_mutants="parent_name",
    value_cols=mic_bac_columns,
    value_preprocessing=None,
    add_relative=True,
)

In [53]:
diff[[f"{bac}_diff" for bac in mic_bac_columns]].values.max()

1385.4333

In [ ]:
import pandas as pd
import numpy as np
from sklearn.utils import resample
from typing import Optional, Generator


def bootstrap_df_generator(
    df: pd.DataFrame,
    n_bootstrap_samples: int,
    bootstrap_group_cols: Optional[list[str]] = None,
    stratify_cols: Optional[list[str]] = None,
    bootstrap_frac: float = 1.0,
    seed: Optional[int] = None,
) -> Generator[pd.DataFrame, None, None]:
    """Yield bootstrap-sampled DataFrames (grouped or rowwise, optional stratification)."""
    rng = np.random.default_rng(seed)
    n_rows = len(df)
    ilocs = np.arange(n_rows)

    # --- determine grouping ---
    if bootstrap_group_cols:
        group_keys = df[bootstrap_group_cols].astype(str).agg("_".join, axis=1)
    else:
        group_keys = pd.Series(ilocs.astype(str), index=df.index)

    # --- construct group table ---
    group_table = (
        pd.DataFrame({"_group_key": group_keys})
        .value_counts()
        .reset_index(name="weight")
    )

    # --- optional stratification ---
    stratify = None
    if stratify_cols:
        stratify = (
            df.groupby(group_keys)[stratify_cols]
            .first()
            .astype(str)
            .agg("_".join, axis=1)
            .reindex(group_table["_group_key"])
            .values
        )

    # --- precompute mapping group → iloc indices ---
    group_to_ilocs = (
        pd.DataFrame({"_group_key": group_keys, "_iloc": ilocs})
        .groupby("_group_key")["_iloc"]
        .apply(np.array)
        .to_dict()
    )

    # --- sampling parameters ---
    n_groups = len(group_table)
    n_samples = int(n_groups * bootstrap_frac)
    weights = group_table["weight"] if bootstrap_group_cols else None

    # --- sampling loop ---
    for _ in range(n_bootstrap_samples):
        rs = int(rng.integers(0, 1e9))
        sampled_groups = resample(
            group_table,
            replace=True,
            n_samples=n_samples,
            stratify=stratify,
            sample_weight=weights,
            random_state=rs,
        )

        iloc_indices = np.concatenate(
            [group_to_ilocs[gk] for gk in sampled_groups["_group_key"]]
        )
        yield df.iloc[iloc_indices].copy()